In [ ]:
import requests
from notecharts import Chart, Palette

# Fetch the Les Misérables dataset
response = requests.get("https://echarts.apache.org/examples/data/asset/data/les-miserables.json")

if response.status_code == 200:
    data = response.json()

# Squish variance in symbolSize
min_size = min(node['symbolSize'] for node in data['nodes'])
max_size = max(node['symbolSize'] for node in data['nodes'])
range_size = max_size - min_size

for node in data['nodes']:
    normalized = (node['symbolSize'] - min_size) / range_size
    node['symbolSize'] = 6 + normalized * 12

# Generate colors and brighter border versions
n_colors = len(data.get("categories", []))
fill_colors = Palette(palette="Plasma", n=n_colors)
border_colors = Palette(palette="Plasma", n=n_colors, value=0.75, saturation=0.4)

# Apply colors to categories
categories = data.get("categories", [])
for i, category in enumerate(categories):
    category["itemStyle"] = {
        "color": fill_colors[i % len(fill_colors)],
        "borderColor": border_colors[i % len(border_colors)],
        "borderWidth": 2
    }

options = {
    "title": {
        "text": "Les Misérables",
        "left": "4%",
        "top": "3%",
        "textStyle": {"fontSize": 18, "fontWeight": 500},
    },
    "textStyle": {
        "fontFamily": "Montserrat"
    },
    "tooltip": {"show": False},
    "toolbox": {
        "feature": {
            "restore": {},
            "saveAsImage": {"pixelRatio": 3},
        }
    },
    "series": [{
        "name": "Les Misérables",
        "type": "graph",
        "layout": "force",
        "data": data['nodes'],
        "links": data['links'],
        "categories": categories,
        "roam": True,
        "draggable": True,
        "label": {
            "show": False,
            "position": "right",
            "fontSize": 11
        },
        "itemStyle": {
            "borderWidth": 3
        },
        "lineStyle": {
            "width": 0.5,
            "opacity": 0.6
        },
        "force": {
            "repulsion": 200,
            "gravity": 0.1,
            "edgeLength": [100, 300],
            "friction": 0.2
        },
    }],
}

Chart(options, width="650px", theme="dark").display()